In [38]:
import random
import json
from pathlib import Path

In [39]:
EPSILON = "ε"

In [40]:
class NFA:
    def __init__(self, states, alphabet, transitions, start_state, accept_states):
        self.states = states
        self.alphabet = alphabet
        self.transitions = transitions or {}
        self.start_state = start_state
        self.accept_states = accept_states

    def add_transition(self, from_state, input_symbol, to_state):
        if from_state not in self.transitions:
            self.transitions[from_state] = {}
        if input_symbol not in self.transitions[from_state]:
            self.transitions[from_state][input_symbol] = set()
        self.transitions[from_state][input_symbol].add(to_state)

    def get_transitions(self, state, input_symbol):
        if state in self.transitions and input_symbol in self.transitions[state]:
            return self.transitions[state][input_symbol]
        return set()

    def is_accepting(self, state):
        return state in self.accept_states

In [41]:
def initialize_nfa(num_states, rng):
    if num_states < 3:
        raise ValueError("Number of states must be at least 3.")

    states = set(range(num_states))
    alphabet = {"0", "1"}

    start_state = 0

    num_accept_states = rng.randint(1, 3)

    accept_states = set(
        rng.sample(list(states), num_accept_states)
    )

    nfa = NFA(
        states=states,
        alphabet=alphabet,
        transitions={},
        start_state=start_state,
        accept_states=accept_states
    )

    return nfa

In [42]:
def get_reachable_states(nfa, start_state = 0):
    reachable = set()
    stack = [start_state]

    while stack:
        current_state = stack.pop()
        if current_state not in reachable:
            reachable.add(current_state)
            for symbol in nfa.alphabet | {EPSILON}:
                for next_state in nfa.get_transitions(current_state, symbol):
                    stack.append(next_state)

    return reachable

In [43]:
def generate_random_transitions(nfa, max_transitions_per_state, rng, allow_epsilon):
    for state in nfa.states:
        num_transitions = rng.randint(1, max_transitions_per_state)
        for _ in range(num_transitions):
            input_symbol = rng.choice(list(nfa.alphabet))
            to_state = rng.choice(list(nfa.states))
            nfa.add_transition(state, input_symbol, to_state)

    if allow_epsilon:
        num_epsilon_transitions = rng.randint(2, 4)
        for _ in range(num_epsilon_transitions):
            from_state = rng.choice(list(nfa.states))
            to_state = rng.choice(list(nfa.states - {from_state}))
            nfa.add_transition(from_state, EPSILON, to_state)

In [44]:
def all_states_reachable(nfa):
    reachable_states = get_reachable_states(nfa, nfa.start_state)

    return reachable_states == nfa.states

In [45]:
def generate_nfa(seed= 0, num_states=10, max_transitions_per_state=3, allow_epsilon=False):
    rng = random.Random(seed)
    max_attempts = 1000

    for _ in range(max_attempts):
        nfa = initialize_nfa(num_states, rng)
        generate_random_transitions(
            nfa, 
            max_transitions_per_state, 
            rng, 
            allow_epsilon
        )

        if all_states_reachable(nfa):
            return nfa

    return RuntimeError("Could not generate a fully reachable NFA.")

In [46]:
def get_nfa_signature(nfa):
    transitions = frozenset(
        (from_state, symbol, to_state)
        for from_state, symbol_dict in nfa.transitions.items()
        for symbol, destinations in symbol_dict.items()
        for to_state in destinations
    )

    return (
        frozenset(nfa.states),
        frozenset(nfa.alphabet),
        nfa.start_state,
        frozenset(nfa.accept_states),
        transitions
    )

## Generate Sparse NFA (upto 3 transitions per state)

In [47]:
sparse_nfas = []
sparse_nfa_signatures = set()

counter = 0
seed = 0

while counter < 30:
    nfa = generate_nfa(seed=seed, num_states=10, max_transitions_per_state=3, allow_epsilon=False)
    signature = get_nfa_signature(nfa)

    if signature not in sparse_nfa_signatures:
        sparse_nfas.append(nfa)
        sparse_nfa_signatures.add(signature)
        counter += 1

    seed += 1


In [48]:
sparse_nfas[0].transitions

{0: {'1': {3, 5, 9}},
 1: {'0': {0}},
 2: {'0': {2}, '1': {1, 8}},
 3: {'1': {2, 9}, '0': {7}},
 4: {'1': {4, 5}, '0': {8}},
 5: {'0': {5, 7, 8}},
 6: {'0': {3}, '1': {5}},
 7: {'1': {5, 6}, '0': {4}},
 8: {'0': {0, 3, 5}},
 9: {'1': {6}}}

## Generate Dense NFA (upto 7 transitions per state)

In [49]:
dense_nfas = []
dense_nfa_signatures = set()

counter = 0
seed = 0

while counter < 30:
    nfa = generate_nfa(seed=seed, num_states=10, max_transitions_per_state=7, allow_epsilon=False)
    signature = get_nfa_signature(nfa)

    if signature not in dense_nfa_signatures:
        dense_nfas.append(nfa)
        dense_nfa_signatures.add(signature)
        counter += 1

    seed += 1


In [50]:
dense_nfas[0].transitions

{0: {'1': {6, 7, 9}},
 1: {'0': {1, 4}},
 2: {'1': {7, 8}, '0': {1, 4, 5}},
 3: {'1': {0, 7, 9}, '0': {8}},
 4: {'0': {1, 3, 5, 8}, '1': {0, 5}},
 5: {'0': {1}, '1': {1, 8}},
 6: {'0': {8, 9}, '1': {8}},
 7: {'1': {5, 7}, '0': {3, 4, 9}},
 8: {'0': {0, 1, 2}, '1': {7, 8}},
 9: {'0': {3, 5, 7}, '1': {3, 5, 7, 9}}}

## Generate NFA with Epsilon Transition

In [51]:
epsilon_nfas = []
epsilon_nfa_signatures = set()

counter = 0
seed = 0

while counter < 30:
    nfa = generate_nfa(seed=seed, num_states=10, max_transitions_per_state=5, allow_epsilon=True)
    signature = get_nfa_signature(nfa)

    if signature not in epsilon_nfa_signatures:
        epsilon_nfas.append(nfa)
        epsilon_nfa_signatures.add(signature)
        counter += 1

    seed += 1

In [52]:
epsilon_nfas[10].transitions

{0: {'0': {3, 8}, '1': {2, 5, 7}},
 1: {'0': {5}},
 2: {'1': {2}},
 3: {'1': {2, 4, 6, 7}, '0': {4}},
 4: {'0': {3, 7}, '1': {0}, 'ε': {2}},
 5: {'1': {3, 6, 8}},
 6: {'0': {3, 9}, '1': {3, 8}, 'ε': {7}},
 7: {'0': {7}, 'ε': {3}},
 8: {'0': {2, 8}, '1': {9}},
 9: {'0': {1, 3, 7}}}

In [53]:
def nfa_to_dict(nfa):
    return {
        "states": sorted(nfa.states),
        "alphabet": sorted(nfa.alphabet),
        "start_state": nfa.start_state,
        "accept_states": sorted(nfa.accept_states),
        "transitions": {
            str(state): {
                symbol: sorted(destinations)
                for symbol, destinations in symbol_dict.items()
            }
            for state, symbol_dict in nfa.transitions.items()
        }
    }

## Exponential-BlowUp NFAs

In [54]:
def generate_blowup_nfas(min_states, max_states):
    rng = random.Random(0)
    blow_up_nfas = []

    for n in range(min_states, max_states + 1):
        blow_up_nfa = initialize_nfa(n, rng)

        blow_up_nfa.add_transition(0, "0", 0)
        blow_up_nfa.add_transition(0, "1", 0)
        blow_up_nfa.add_transition(0, "1", 1)
        for i in range(1, n):
            blow_up_nfa.add_transition(i, "0", i + 1)
            blow_up_nfa.add_transition(i, "1", i + 1)

        blow_up_nfa.accept_states = {n}

        blow_up_nfas.append(blow_up_nfa)

    return blow_up_nfas

In [55]:
blowup_nfas = generate_blowup_nfas(4, 10)

## Save NFAs

In [57]:
output_dir = Path("data/generated")
output_dir.mkdir(parents=True, exist_ok=True)

data = [nfa_to_dict(nfa) for nfa in sparse_nfas]

with open(output_dir / "sparse_nfas.json", "w") as file:
    json.dump(data, file, indent=4)

data = [nfa_to_dict(nfa) for nfa in dense_nfas]

with open(output_dir / "dense_nfas.json", "w") as file:
    json.dump(data, file, indent=4)

data = [nfa_to_dict(nfa) for nfa in epsilon_nfas]

with open(output_dir / "epsilon_nfas.json", "w") as file:
    json.dump(data, file, indent=4)

with open(output_dir / "blowup_nfas.json", "w") as file:
    json.dump([nfa_to_dict(nfa) for nfa in blowup_nfas], file, indent=4)